## 1. Generating Synthetic Financial Transactions

To simulate a real-world scenario, we'll generate 1,000,000 individual financial transactions. Each transaction will be a random dollar amount between $0.01 and $100.00, rounded to two decimal places. These will be our "ground truth" values that both `float` and `Decimal` sums will attempt to aggregate.

In [ ]:
# Import the random module for generating synthetic data
import random

# Define the number of transactions as seen in the video
NUM_TRANSACTIONS = 1_000_000

# Generate 1,000,000 synthetic financial transactions.
# Each is a float, rounded to two decimal places, representing a dollar amount.
# This list serves as the *exact* set of numbers we want to sum accurately.
raw_transaction_values = [
    round(random.uniform(0.01, 100.00), 2)
    for _ in range(NUM_TRANSACTIONS)
]

# Print a sample of the generated transactions and their total count
print(f"Generated {len(raw_transaction_values):,} raw transaction values.")
print(f"Sample transactions: {raw_transaction_values[:5]}...")
print(f"Each value is a standard Python float, rounded to two decimal places.")

## 2. The Naive Attempt: Summing with Standard Floats

When we sum these transactions using Python's built-in `float` type, we expect a precise total. However, due to the way `float` numbers are stored in binary, tiny inaccuracies accumulate, leading to a total that is subtly, yet consistently, off from the mathematically perfect sum. This is the "missing pennies" problem.

In [ ]:
# Import the Decimal module and getcontext for precise arithmetic
from decimal import Decimal, getcontext

# Set the precision for Decimal operations. 28 is common for financial calculations
# to handle values up to hundreds of trillions with 2 decimal places.
getcontext().prec = 28

# --- Calculate the 'perfect' sum using Decimal for comparison ---
# To get a truly perfect sum, we convert each raw float value to a Decimal
# by first converting it to a string. This avoids intermediate float inaccuracies.
perfect_total_decimal = sum(
    Decimal(str(val)) for val in raw_transaction_values
)

# --- Calculate the sum using standard Python floats ---
# This directly sums the float values we generated. This is where precision errors occur.
total_float = sum(raw_transaction_values)

# Calculate the difference between the float sum and the perfect Decimal sum.
# We convert the float sum back to a Decimal for an accurate difference calculation.
difference_float = perfect_total_decimal - Decimal(str(total_float))

# Print the results, formatted to two decimal places for financial context
print(f"Float total: {total_float:.2f}")
print(f"Perfect total: {perfect_total_decimal:.2f}")
print(f"Difference (Float vs. Perfect): {difference_float:.2f}")
print(
    "As you can see, even with values rounded to two decimal places, "
    "the float sum drifts from the perfect sum."
)

## 3. Understanding Float Precision Limitations

This isn't a bug in Python; it's a fundamental aspect of how computers store floating-point numbers. Standard `float` types use a binary (base-2) representation. Just as 1/3 cannot be represented exactly in base-10 (0.333...), numbers like 0.1 or 0.7 cannot be represented exactly in base-2. They become repeating fractions, which are then truncated, leading to tiny errors that accumulate.

In [ ]:
# Demonstrate how a simple decimal like 0.1 is stored as an approximation in binary floats
# This shows the actual value a float holds for 0.1
print(f"0.1 as a standard float, to 20 decimal places: {0.1:.20f}")
print(
    "Notice the trailing '555' at the end. This tiny error is stored and accumulates "
    "when many such numbers are added together."
)

## 4. The Solution: Python's `Decimal` Module

For financial calculations where absolute precision is paramount, Python offers the `Decimal` module. It forces the computer to perform calculations using base-10 (decimal) arithmetic, just like humans do. This ensures that every digit, especially after the decimal point, is perfectly represented and accounted for.

In [ ]:
# The Decimal module was already imported and precision set in a previous step.
# Here, we demonstrate how to explicitly convert a value to a Decimal object.
# It's crucial to convert from a string representation of the number to avoid
# intermediate float inaccuracies during the Decimal object creation.

value_float = 5.37
value_decimal = Decimal(str(value_float))

print(f"Original float value: {value_float}")
print(f"Converted Decimal value: {value_decimal}")
print(f"Type of float value: {type(value_float)}")
print(f"Type of Decimal value: {type(value_decimal)}")
print(
    "Using Decimal(str(value)) ensures that the exact decimal value is preserved "
    "during conversion, avoiding any float-related approximation."
)

## 5. Solving the Precision Problem with `Decimal`

Now, we'll re-sum the *exact same* set of 1,000,000 transactions, but this time, we'll convert each value to a `Decimal` object before summing. This guarantees a mathematically perfect total, matching our `perfect_total_decimal` exactly.

In [ ]:
# The Decimal module was already imported and precision set.

# --- Calculate the sum using Python's Decimal module ---
# Each raw float value is converted to a Decimal via its string representation
# before being added. This ensures decimal precision throughout the summation.
total_decimal = sum(
    Decimal(str(val)) for val in raw_transaction_values
)

# Calculate the difference between the Decimal sum and the perfect Decimal sum.
# This difference should ideally be zero.
difference_decimal = perfect_total_decimal - total_decimal

# Print the results, formatted to two decimal places for financial context
print(f"Decimal total: {total_decimal:.2f}")
print(f"Perfect total: {perfect_total_decimal:.2f}")
print(f"Difference (Decimal vs. Perfect): {difference_decimal:.2f}")
print(
    "With the Decimal module, the sum perfectly matches the mathematically "
    "calculated total, solving the precision problem."
)

## 6. Understanding the Trade-off: Memory Footprint

While `Decimal` provides perfect precision, it's not without cost. `Decimal` objects are more complex than native `float` types and thus consume significantly more memory. This trade-off is important to consider, especially when dealing with extremely large datasets where memory optimization is critical.

In [ ]:
# Import the sys module to inspect object sizes
import sys

# Get the size of a standard float object (e.g., 0.1)
float_size = sys.getsizeof(0.1)

# Get the size of a Decimal object initialized from a string (e.g., '0.1')
decimal_size = sys.getsizeof(Decimal('0.1'))

print(f"Memory footprint of a float (0.1): {float_size} bytes")
print(f"Memory footprint of a Decimal('0.1'): {decimal_size} bytes")
print(
    "As you can see, a Decimal object typically takes up more memory than a float. "
    "Multiply this difference by millions or billions of data points, and the "
    "memory usage can become substantial."
)

## Conclusion: Precision for Financial Integrity

You've successfully seen how standard floating-point numbers can introduce subtle errors in financial calculations and how Python's `Decimal` module provides an accurate, reliable solution. For applications where every penny counts, `Decimal` is the clear choice.

While this notebook didn't generate a separate artifact file, the key takeaway is the accurate `Decimal` total printed above. Remember the trade-off: `Decimal` objects consume more memory than `float`s, a factor to consider in performance-critical or memory-constrained applications. We'll explore strategies for managing this in future discussions.